# Module 2.2: Hardening SSH & Baseline System Configuration

## Homelab NAS Development - Step 2: Home Network Connection

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Establish passwordless Ed25519 SSH key authentication between workstation and Raspberry Pi.
2. Understand server privilege separation and SSH security hardening principles.
3. Harden SSH daemon configuration against brute-force attacks.
4. Update system packages and install essential administrative utilities.

## 2.2.1 Conceptual Background: Server Hardening & Privilege Separation

SSH (Secure Shell) is the gateway to your headless server. Securing this entry point is vital, especially when you plan to connect remote clients or VPNs. 

### Why Ed25519 Keys?
Modern cryptographic standards recommend **Ed25519** (Elliptic Curve 25519) over legacy RSA 2048/4096. Ed25519 provides superior security, resistance to timing attacks, smaller key sizes (68 characters vs. over 500 for RSA), and faster handshake negotiation.

### Privilege Separation Principles
Logging in directly as `root` bypasses command auditing and exposes the system to catastrophic typos or automated dictionary attacks. The standard Linux security posture disables direct root login (`PermitRootLogin no`) and password authentication (`PasswordAuthentication no`). Administrators log in as a dedicated user and elevate privileges via `sudo`, creating a timestamped audit trail in `/var/log/auth.log`.

## 2.2.2 Step-by-Step Laboratory Guide

### Task 2.2.1: Generate and Copy SSH Keys

To adhere to security best practices and credential isolation, generate a dedicated Ed25519 keypair for your homelab so it remains distinct from any personal, work, or cloud credentials.

From your workstation terminal:

#### 1. Generate a Dedicated Keypair (with Passphrase)
Run `ssh-keygen` with the `-f` flag to specify the exact path and filename so existing keys are not overwritten.

> [!TIP]
> **Security Best Practice: Passphrase Protection**
> Always configure a strong passphrase when prompted. Setting a passphrase encrypts the private key file at rest using AES-256. Even if your workstation disk or backup is compromised, an attacker cannot utilize your key without cracking the passphrase.

```bash
# Generate a dedicated Ed25519 keypair
ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519_homelab -C "homelab-access"
```
When prompted:
```text
Enter passphrase (empty for no passphrase): <Enter your secure passphrase>
Enter same passphrase again: <Confirm passphrase>
```

#### 2. Enable the Workstation SSH Agent Service
To avoid typing your passphrase every single time you connect, configure the workstation's background SSH Agent service. Once running, the agent caches the decrypted key in memory for your session.

- **Windows (Run once in an Administrator PowerShell):**
  Windows includes OpenSSH Agent, but it is disabled by default. Enable and start it:
  ```powershell
  Set-Service -Name ssh-agent -StartupType Automatic
  Start-Service ssh-agent
  ```
- **macOS / Linux / Git Bash:**
  macOS runs `ssh-agent` natively. On Linux or Git Bash:
  ```bash
  eval $(ssh-agent -s)
  ```

#### 3. Configure SSH Client Config (`~/.ssh/config`) & Session Key Caching
Configure your workstation's SSH client configuration file located at `~/.ssh/config` (on Windows: `C:\Users\<Username>\.ssh\config`).

By including `AddKeysToAgent yes`, SSH will prompt for your passphrase **only on the first login of your session**, automatically load the decrypted key into your running `ssh-agent`, and allow all subsequent connections to authenticate seamlessly without prompting.

> **File Location & Appending:** 
> - If `~/.ssh/config` already exists (e.g., containing host configurations for GitHub or work servers), scroll to the **very bottom of the file**, add a blank line, and append the block below.
> - If the file does **not** exist yet, create it as a plain text file named `config` with **no file extension** (do not let Windows save it as `config.txt`).

**Open or create the config file from your workstation:**
- **Windows (PowerShell):**
  ```powershell
  # Opens or creates ~/.ssh/config in Notepad
  notepad $HOME\.ssh\config
  ```
- **macOS / Linux / Git Bash:**
  ```bash
  nano ~/.ssh/config
  ```

**Add the following configuration block to the bottom of the file:**
```text
Host homelab-pi
    HostName homelab-pi.local
    User pi
    IdentityFile ~/.ssh/id_ed25519_homelab
    IdentitiesOnly yes
    AddKeysToAgent yes
```

**Understanding the Directives:**
- `Host homelab-pi`: Defines a local shortcut alias. You can now simply run `ssh homelab-pi`.
- `HostName homelab-pi.local`: Specifies the actual mDNS hostname (or static IPv4 address) of your Raspberry Pi.
- `User pi`: Automatically provides the remote Linux username.
- `IdentityFile ~/.ssh/id_ed25519_homelab`: Tells the SSH client exactly which private key to present for this host.
- `IdentitiesOnly yes`: Enforces that SSH only attempts the specified identity file, preventing "Too many authentication failures" errors if you have multiple keys loaded into an SSH agent.
- `AddKeysToAgent yes`: Automatically caches the decrypted key into `ssh-agent` upon your first successful login, eliminating subsequent passphrase prompts for the duration of your session.

#### 4. Copy the Public Key to the Raspberry Pi
Next, transfer your newly created **public key** (`.pub`) into the Raspberry Pi's authorized keys list. Choose the command matching your workstation operating system:

**Windows (PowerShell):**
```powershell
Get-Content ~/.ssh/id_ed25519_homelab.pub | ssh pi@homelab-pi.local "mkdir -p ~/.ssh && chmod 700 ~/.ssh && cat >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys"
```

**macOS / Linux / Git Bash:**
```bash
ssh-copy-id -i ~/.ssh/id_ed25519_homelab.pub pi@homelab-pi.local
```

### Task 2.2.2: Hardening the SSH Daemon

Log in to your Raspberry Pi using your configured shortcut or target hostname:
```bash
ssh homelab-pi
```

Edit the drop-in SSH server configuration file:
```bash
sudo nano /etc/ssh/sshd_config.d/homelab.conf
```

Add the following security constraints:
```text
PermitRootLogin no
PasswordAuthentication no
PubkeyAuthentication yes
AuthorizedKeysFile .ssh/authorized_keys
```

Restart the SSH service to apply configuration:
```bash
sudo systemctl restart ssh
```

> [!CAUTION]
> **Lockout Prevention (Critical Safety Precaution):**
> Do **NOT** close your active terminal session after restarting the SSH service! 
> Open a **new, separate terminal window** on your workstation and test logging in with your SSH key (`ssh homelab-pi` or `ssh pi@homelab-pi.local`) to verify key authentication works before exiting your existing session.
>
> If you make a syntax error or your key was not properly copied, password login will be disabled. Keeping your current session active ensures you can immediately edit `/etc/ssh/sshd_config.d/homelab.conf` to rectify any issues without losing access.

### 🖼️ Task 2.2.2 Verification: Successful SSH Key Authentication & Login Banner

**Verification Objective:** Attach a terminal screenshot demonstrating successful passwordless SSH login into your Raspberry Pi (`ssh homelab-pi`) authenticated via your Ed25519 key, displaying the standard Debian Linux system banner without requesting an account password.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 2.2.3: Baseline Package Upgrade & System Tools

From your active SSH session on the Raspberry Pi, synchronize repository package indexes, upgrade all existing packages to their newest releases, and install standard administration utilities:

```bash
sudo apt update && sudo apt full-upgrade -y
sudo apt install -y htop tmux curl git smartmontools ufw
```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Dedicated Ed25519 SSH keypair generated and client config created on workstation.
- [ ] SSH daemon hardened (`PasswordAuthentication no`, `PermitRootLogin no`) and verified in a separate session.
- [ ] Verification screenshot of successful SSH key login captured and attached directly above.
- [ ] System packages upgraded and foundational administrative utilities installed.

Proceed to the next module in sequence once this checkpoint is satisfied.